In [353]:
import pandas as pd
import requests
import json

# **API REQUETE GLOBALE** #

In [354]:
# CALL API
API_KEY = "haj00y9FzSQ1VZCTUjpmla8Q98xRnA6a"

URL = 'https://prim.iledefrance-mobilites.fr/marketplace/disruptions_bulk/disruptions/v2'

headers = {
    "Accept": "application/json",
    "apikey": API_KEY
}

response = requests.get(URL, headers=headers)

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Erreur {response.status_code}: {response.text}")

{'disruptions': [{'id': '74316236-8627-11ef-aca5-0a58a9feac02', 'applicationPeriods': [{'begin': '20241012T020900', 'end': '20241013T233400'}, {'begin': '20241019T120900', 'end': '20241020T230900'}, {'begin': '20241207T020900', 'end': '20241208T234400'}, {'begin': '20250118T020900', 'end': '20250119T234900'}, {'begin': '20250329T020900', 'end': '20250330T235400'}, {'begin': '20250405T030400', 'end': '20250406T234900'}, {'begin': '20250412T030400', 'end': '20250413T234900'}], 'lastUpdate': '20250401T120646', 'cause': 'TRAVAUX', 'severity': 'BLOQUANTE', 'tags': ['Actualité'], 'title': 'Travaux boulevard Carnot et Calmette', 'message': '<p>En raison de travaux boulevard Carnot et Calmette à Mantes-la-Jolie, les arrêts suivants ne seront pas desservis :</p><p>Ligne D : (Plaisances) Carnot reporté à Louise Michel et Calmette reporté à Mantes-la-Ville Mairie</p><p>Ligne N : Calmette et Carnot reportés à Mantes-Station</p><p>Ligne A14 : Calmette reporté à Mantes-Station</p><p>les samedis 05 e

## **Récupérer les données présente dna sles clés API** ##

In [355]:
# Accéder aux clés des perturbations
disruptions = data.get("disruptions", [])

In [356]:
# Accéder aux clés des arrêts concernés
lines = data.get("lines", [])

In [357]:
lines

[{'id': 'line:IDFM:C01423',
  'name': '1',
  'shortName': '1',
  'mode': 'Bus',
  'networkId': 'network:IDFM:6',
  'impactedObjects': [{'type': 'line',
    'id': 'line:IDFM:C01423',
    'name': '1',
    'disruptionIds': ['4884f0e6-00cd-11f0-af9a-0a58a9feac02']},
   {'type': 'stop_point',
    'id': 'stop_point:IDFM:16820',
    'name': 'Rond-Point des Sciences',
    'disruptionIds': ['4884f0e6-00cd-11f0-af9a-0a58a9feac02']},
   {'type': 'stop_point',
    'id': 'stop_point:IDFM:16819',
    'name': 'Rond-Point des Sciences',
    'disruptionIds': ['4884f0e6-00cd-11f0-af9a-0a58a9feac02']},
   {'type': 'stop_point',
    'id': 'stop_point:IDFM:19398',
    'name': 'Ampère',
    'disruptionIds': ['4884f0e6-00cd-11f0-af9a-0a58a9feac02']}]},
 {'id': 'line:IDFM:C02129',
  'name': '3',
  'shortName': '3',
  'mode': 'Bus',
  'networkId': 'network:IDFM:6',
  'impactedObjects': [{'type': 'line',
    'id': 'line:IDFM:C02129',
    'name': '3',
    'disruptionIds': ['5f95054a-d707-11ef-b937-0a58a9feac02',

## **Convertir les data de la clé diruption en Dataframe** ##

In [358]:
pd.reset_option('display.max_colwidth')

# Normalisation avec l'option errors='ignore' pour ignorer les clés manquantes
df_disruptions = pd.json_normalize(data['disruptions'], 
                                  record_path='applicationPeriods', 
                                  meta=['id', 'lastUpdate', 'cause', 'severity', 'tags', 'title', 'message', 'shortMessage'], 
                                  sep=',', errors='ignore')

# Affichage du résultat
df_disruptions.head()

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage
0,20241012T020900,20241013T233400,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN
1,20241019T120900,20241020T230900,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN
2,20241207T020900,20241208T234400,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN
3,20250118T020900,20250119T234900,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN
4,20250329T020900,20250330T235400,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN


## **Convertir les data de la clé lines en Dataframe** ##

In [359]:
from collections import defaultdict

# 1. Créer un dictionnaire pour lier disruptionId aux arrêts concernés
disruption_to_stops = defaultdict(set)

# 2. Remplir ce dictionnaire et ajouter les informations 'name' et 'mode' à chaque disruptionId
disruption_line_info = {}  # Dictionnaire pour lier disruptionId aux informations de ligne (name, mode)

for line in lines:
    for obj in line.get('impactedObjects', []):
        if obj['type'] == 'stop_point':  # Filtrer uniquement les objets de type 'stop_point'
            stop_name = obj['name']
            for disruption_id in obj.get('disruptionIds', []):
                # Ajouter l'arrêt à la liste des arrêts pour ce disruptionId
                disruption_to_stops[disruption_id].add(stop_name)
                # Ajouter les informations de la ligne (name, mode) pour ce disruptionId
                if disruption_id not in disruption_line_info:
                    disruption_line_info[disruption_id] = {
                        'name': line['name'],
                        'mode': line['mode']
                    }

# 3. Construction du DataFrame avec les arrêts pour chaque disruptionId
df_lines = pd.DataFrame([
    {
        'disruptionId': disruption_id,
        'stop_points': sorted(list(stops)),
        'name': disruption_line_info[disruption_id]['name'],
        'mode': disruption_line_info[disruption_id]['mode'],
        
    }
    for disruption_id, stops in disruption_to_stops.items()
])

# Affichage du DataFrame final
df_lines.head()

,disruptionId,stop_points,name,mode
0,4884f0e6-00cd-11f0-af9a-0a58a9feac02,"[Ampère, Rond-Point des Sciences]",1,Bus
1,5f95054a-d707-11ef-b937-0a58a9feac02,"[Collège Maria Callas, Désiré Lefèvre, Le Chat...",3,Bus
2,2747d1da-11c3-11f0-b7af-0a58a9feac02,"[Carrefour du 19 Mars 1962, Petit-Châtenay, Ru...",412,Bus
3,31eec03a-11c3-11f0-bb7e-0a58a9feac02,"[Carrefour du 19 Mars 1962, Petit-Châtenay, Ru...",412,Bus
4,ff5f1896-d80a-11ef-b2cf-0a58a9feac02,"[Place Foch, Rue Andin]",5206 (ex P),Bus


## **Filtrer évènements en cours du dataset disruptions** ##

In [360]:
from datetime import datetime

def filtrer_evenements_en_cours(df, col_debut="begin", col_fin="end", now=None):
    """
    Filtre les événements en cours à partir d'un DataFrame avec colonnes 'start' et 'end' (format 'YYYYMMDDTHHMMSS').

    :param df: DataFrame contenant les événements
    :param col_debut: nom de la colonne de début (par défaut "start")
    :param col_fin: nom de la colonne de fin (par défaut "end")
    :param now: datetime personnalisé (utile pour les tests), sinon datetime.now()
    :return: DataFrame filtré avec les événements en cours
    """
    df = df.copy()
    
    # Conversion des dates
    df[col_debut] = pd.to_datetime(df[col_debut], format="%Y%m%dT%H%M%S", errors='coerce')
    df[col_fin] = pd.to_datetime(df[col_fin], format="%Y%m%dT%H%M%S", errors='coerce')

    # Date/heure actuelle
    now = now or pd.Timestamp.now()

    # Filtrage des événements en cours
    df_en_cours = df[(df[col_debut] <= now) & (df[col_fin] >= now)]
    
    return df_en_cours


In [361]:
df_en_cours = filtrer_evenements_en_cours(df_disruptions)
df_en_cours.head()

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage
5,2025-04-05 03:04:00,2025-04-06 23:49:00,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN
35,2025-01-06 01:00:00,2025-06-06 23:00:00,4a2946e2-beae-11ef-b1c7-0a58a9feac02,20241220T094303,TRAVAUX,BLOQUANTE,NaN,"4414, 4438 et TàD Etréchy, Travaux Chauffour-l...","<p>Lignes 4414, 4438 et TàD Etréchy, du 6/01/2...",NaN
542,2025-01-13 00:00:00,2025-12-31 23:59:00,30376efa-cea6-11ef-8d37-0a58a9feac02,20250109T172424,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,<p>En raison de travaux avenue Jean Jaurès à M...,NaN
543,2025-01-13 00:00:00,2025-06-30 23:59:00,426b148c-ceab-11ef-8d37-0a58a9feac02,20250313T203153,TRAVAUX,BLOQUANTE,NaN,🚧 5121 5152 - Travaux : Arrêt Golf National da...,<p><strong>🚧 #Perturbations #Ligne5121 #Ligne5...,NaN
544,2025-01-12 00:31:00,2025-12-24 00:31:00,dead0f6a-d079-11ef-b1c7-0a58a9feac02,20250112T011211,TRAVAUX,BLOQUANTE,NaN,Bus N21 : Travaux - Arrêt(s) non desservi(s),<p>La ligne N21 est déviée : les arrêts situés...,NaN


## **Merged dataset disruptions et lines sur les clés communes disruptionId et id** ##

In [362]:
# On renomme "disruptionId" en "id" dans df_lines pour pouvoir faire la jointure facilement
df_lines_renamed = df_lines.rename(columns={'disruptionId': 'id'})

# Merge (left join) : on garde tout df_en_cours, et on ajoute les colonnes de df_lines si id commun
df_merged = df_en_cours.merge(df_lines_renamed, on='id', how='left')

## **PREPROCESSING** ##

In [363]:
from bs4 import BeautifulSoup

# Supprimer valeurs NAN et converttir en str
df_merged['message'] = df_merged['message'].astype(str).fillna('')

# Appliquer la suppression des balises HTML à chaque message de la colonne 'message'
df_merged['message'] = df_merged['message'].apply(
    lambda msg: BeautifulSoup(msg, "html.parser").get_text()
)


In [364]:
df_merged.head()

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode
0,2025-04-05 03:04:00,2025-04-06 23:49:00,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,En raison de travaux boulevard Carnot et Calme...,NaN,NaN,NaN,NaN
1,2025-01-06 01:00:00,2025-06-06 23:00:00,4a2946e2-beae-11ef-b1c7-0a58a9feac02,20241220T094303,TRAVAUX,BLOQUANTE,NaN,"4414, 4438 et TàD Etréchy, Travaux Chauffour-l...","Lignes 4414, 4438 et TàD Etréchy, du 6/01/2025...",NaN,[Grande Rue],4414,Bus
2,2025-01-13 00:00:00,2025-12-31 23:59:00,30376efa-cea6-11ef-8d37-0a58a9feac02,20250109T172424,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,En raison de travaux avenue Jean Jaurès à Mant...,NaN,[Poste],21,Bus
3,2025-01-13 00:00:00,2025-06-30 23:59:00,426b148c-ceab-11ef-8d37-0a58a9feac02,20250313T203153,TRAVAUX,BLOQUANTE,NaN,🚧 5121 5152 - Travaux : Arrêt Golf National da...,🚧 #Perturbations #Ligne5121 #Ligne5152📅 à part...,NaN,[Golf National],5121,Bus
4,2025-01-12 00:31:00,2025-12-24 00:31:00,dead0f6a-d079-11ef-b1c7-0a58a9feac02,20250112T011211,TRAVAUX,BLOQUANTE,NaN,Bus N21 : Travaux - Arrêt(s) non desservi(s),La ligne N21 est déviée : les arrêts situés en...,NaN,"[Barbusse - Larroumes, Carrefour des Poulets, ...",N21,Bus


In [365]:
# Récupérer les ID des perturbations du précédent appel API
# Par exemple, ces ID pourraient être stockés dans un fichier ou une base de données.
previous_disruptions = set(df_merged['id'].tolist())

# Simulons un nouvel appel API en modifiant df_merged pour représenter le nouvel état des perturbations
# Ce dataframe (df_merged) serait mis à jour à chaque appel API avec de nouvelles données.
new_disruptions = set(df_merged['id'].tolist())  # Ici on reprend la même liste, mais dans un vrai cas, ce serait mis à jour.

# Ajouter la colonne 'status' en fonction des conditions
df_merged['status'] = df_merged['id'].apply(
    lambda x: 'new' if x not in previous_disruptions else ('finished' if x not in new_disruptions else 'now')
)

# Afficher le DataFrame avec la nouvelle colonne 'status'
df_merged.head()

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode,status
0,2025-04-05 03:04:00,2025-04-06 23:49:00,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,En raison de travaux boulevard Carnot et Calme...,NaN,NaN,NaN,NaN,now
1,2025-01-06 01:00:00,2025-06-06 23:00:00,4a2946e2-beae-11ef-b1c7-0a58a9feac02,20241220T094303,TRAVAUX,BLOQUANTE,NaN,"4414, 4438 et TàD Etréchy, Travaux Chauffour-l...","Lignes 4414, 4438 et TàD Etréchy, du 6/01/2025...",NaN,[Grande Rue],4414,Bus,now
2,2025-01-13 00:00:00,2025-12-31 23:59:00,30376efa-cea6-11ef-8d37-0a58a9feac02,20250109T172424,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,En raison de travaux avenue Jean Jaurès à Mant...,NaN,[Poste],21,Bus,now
3,2025-01-13 00:00:00,2025-06-30 23:59:00,426b148c-ceab-11ef-8d37-0a58a9feac02,20250313T203153,TRAVAUX,BLOQUANTE,NaN,🚧 5121 5152 - Travaux : Arrêt Golf National da...,🚧 #Perturbations #Ligne5121 #Ligne5152📅 à part...,NaN,[Golf National],5121,Bus,now
4,2025-01-12 00:31:00,2025-12-24 00:31:00,dead0f6a-d079-11ef-b1c7-0a58a9feac02,20250112T011211,TRAVAUX,BLOQUANTE,NaN,Bus N21 : Travaux - Arrêt(s) non desservi(s),La ligne N21 est déviée : les arrêts situés en...,NaN,"[Barbusse - Larroumes, Carrefour des Poulets, ...",N21,Bus,now


## **Ouvrir le précédent appel API s'il existe** ##

In [366]:
# Charger le fichier CSV de la version précédente
previous_file = "df_previous_merged.csv"

try:
    df_previous = pd.read_csv(previous_file)
    print("Fichier précédent chargé depuis CSV.")
except FileNotFoundError:
    df_previous = pd.DataFrame(columns=["id", "name", "mode", "begin", "end", "severity", "tags", "title", "message", "status"])
    print("Aucun fichier précédent trouvé, c’est le premier appel ?")

Fichier précédent chargé depuis CSV.


## **Remplacer l'ancien appel API par l'actuel qui deviendra lui même l'ancien appel API** ##

In [367]:
df_merged.to_csv(f"df_previous_merged.csv", index=False)

In [368]:
df_current = df_merged

## **Mise à jour du statut des disruptions : nouvelles, en cours, terminées** ##


In [369]:
# Ajouter la colonne 'status' pour les disruptions actuelles
df_current["status"] = df_current["id"].apply(
    lambda x: "new" if x not in df_previous["id"].values else "now"
)

# Identifier les disruptions terminées ("finished")
finished_ids = set(df_previous["id"]) - set(df_current["id"])
df_finished = df_previous[df_previous["id"].isin(finished_ids)].copy()
df_finished["status"] = "finished"  # Ajouter le statut "finished"

## **Concaténer les résultats** ##

In [370]:
# Fusionner les disruptions actuelles et terminées
df_all = pd.concat([df_current, df_finished], ignore_index=True)


In [371]:
df_all.head()

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode,status
0,2025-04-05 03:04:00,2025-04-06 23:49:00,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,En raison de travaux boulevard Carnot et Calme...,NaN,NaN,NaN,NaN,now
1,2025-01-06 01:00:00,2025-06-06 23:00:00,4a2946e2-beae-11ef-b1c7-0a58a9feac02,20241220T094303,TRAVAUX,BLOQUANTE,NaN,"4414, 4438 et TàD Etréchy, Travaux Chauffour-l...","Lignes 4414, 4438 et TàD Etréchy, du 6/01/2025...",NaN,[Grande Rue],4414,Bus,now
2,2025-01-13 00:00:00,2025-12-31 23:59:00,30376efa-cea6-11ef-8d37-0a58a9feac02,20250109T172424,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,En raison de travaux avenue Jean Jaurès à Mant...,NaN,[Poste],21,Bus,now
3,2025-01-13 00:00:00,2025-06-30 23:59:00,426b148c-ceab-11ef-8d37-0a58a9feac02,20250313T203153,TRAVAUX,BLOQUANTE,NaN,🚧 5121 5152 - Travaux : Arrêt Golf National da...,🚧 #Perturbations #Ligne5121 #Ligne5152📅 à part...,NaN,[Golf National],5121,Bus,now
4,2025-01-12 00:31:00,2025-12-24 00:31:00,dead0f6a-d079-11ef-b1c7-0a58a9feac02,20250112T011211,TRAVAUX,BLOQUANTE,NaN,Bus N21 : Travaux - Arrêt(s) non desservi(s),La ligne N21 est déviée : les arrêts situés en...,NaN,"[Barbusse - Larroumes, Carrefour des Poulets, ...",N21,Bus,now


In [376]:
df_all['mode'].value_counts()

mode
Bus             232
RapidTransit      4
LocalTrain        3
Tramway           1
Metro             1
Name: count, dtype: int64

## **Filtrer les disruptions concernant le réseau férré** ##

In [ ]:
df_all_filtered = df_all[df_all['mode'].isin(['RapidTransit', 'LocalTrain', 'Tramway', 'Metro'])]

In [380]:
df_all_filtered.shape

(9, 14)

In [379]:
df_all_filtered.head(10)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode,status
47,2025-04-05 23:30:00,2025-04-07 02:30:00,7750a9c6-0594-11f0-8f00-0a58a9feac02,20250320T150606,TRAVAUX,BLOQUANTE,[Actualité],RER C : Auster.-Choisy-Juvisy 05/04 au 06/04.,Période : à partir de 23h45 jusqu’en fin de so...,Trafic interrompu planifié,"[Ablon, Arpajon, Athis-Mons, Bibliothèque Fran...",C,RapidTransit,now
48,2025-04-06 03:00:00,2025-04-07 03:00:00,cff4d016-0594-11f0-949f-0a58a9feac02,20250320T150835,TRAVAUX,BLOQUANTE,[Actualité],RER C: Massy P.- Pt de Rungis 6-12-13-26-27 avril,Période : Toute la journée.Date : le dimanche...,Trafic interrompu planifié,"[Bibliothèque François Mitterrand, Chemin d'An...",C,RapidTransit,now
73,2025-03-27 18:42:00,2025-04-22 04:30:00,26847bb8-0b33-11f0-b7af-0a58a9feac02,20250327T184436,TRAVAUX,BLOQUANTE,[Actualité],Tramway T1 : Travaux - Trafic interrompu,"Jusqu'au lundi 21 avril 2025, le trafic est in...",Trafic interrompu,"[Auguste Delaune, Bobigny - Pablo Picasso, Esc...",T1,Tramway,now
87,2025-04-06 03:00:00,2025-04-07 03:00:00,29d01dd4-0edc-11f0-8f48-0a58a9feac02,20250401T113201,TRAVAUX,BLOQUANTE,[Actualité],ligne N : Bellevue non desservi du 05 au 06/04,Période : du premier au dernier trainDates : d...,Arrêt non desservi planifié,"[Chaville Rive Gauche, Clamart, Coignières, Dr...",N,LocalTrain,now
88,2025-04-06 03:00:00,2025-04-07 03:00:00,171a3cca-0ee0-11f0-ac05-0a58a9feac02,20250401T120007,TRAVAUX,BLOQUANTE,[Actualité],RER C : Choisy-Juvisy 05/04 au 06/04.,Période : Tout le week-end.Dates : Samedi 5 et...,Arrêts non desservis planifiés,"[Ablon, Arpajon, Athis-Mons, Bouray, Breuillet...",C,RapidTransit,now
143,2025-04-06 03:00:00,2025-04-07 03:00:00,f08ac200-11cd-11f0-b7af-0a58a9feac02,20250405T052745,TRAVAUX,BLOQUANTE,[Actualité],Ligne V : Versailles-Chantiers / Massy P 5-6-1...,Période : Tout le week-end.Dates : les week-en...,Trafic interrompu planifié,"[Bièvres, Igny, Jouy-en-Josas, Massy - Palaise...",V,LocalTrain,now
247,2025-04-06 20:09:06,2025-04-06 21:45:00,38cc8a04-1317-11f0-88c2-0a58a9feac02,20250406T204450,PERTURBATION,BLOQUANTE,[Actualité],Ligne N : Dreux <-> Plaisir - Grignon trafic i...,Le trafic est interrompu entre Dreux et Plaisi...,interruption,"[Chaville Rive Gauche, Clamart, Dreux, Fontena...",N,LocalTrain,now
364,2025-04-06 20:11:22,2025-04-06 22:45:00,1be00b94-1314-11f0-88c2-0a58a9feac02,20250406T202234,PERTURBATION,PERTURBEE,[Actualité],RER E : perturbations,Le trafic est perturbé entre Nanterre-La-Folie...,fortement ralenti,"[Gretz-Armainvilliers, Haussmann Saint-Lazare,...",E,RapidTransit,now
366,2025-04-06 21:16:41,2025-04-06 21:40:04,2a1c30b8-131c-11f0-bb7e-0a58a9feac02,20250406T212013,PERTURBATION,PERTURBEE,[Actualité],Métro 10 : Panne de portes de train - Train st...,Un train stationne à Boulogne Jean Jaurès en d...,Train stationne,[Boulogne Jean Jaurès],10,Metro,new


In [374]:
df_all.status.value_counts()

status
now         453
new           1
finished      1
Name: count, dtype: int64

Comment lire les indications temporelles

begin 20241012T020900

2024 : Année

10 : Mois (octobre)

12 : Jour

T : Séparateur entre la date et l'heure (utilisé pour indiquer que ce qui suit est l'heure)

02 : Heure (2 heures du matin)

09 : Minute

00 : Seconde